In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from arch import arch_model
import itertools
import warnings
from scipy import stats
from matplotlib import pyplot as plt


In [2]:
print("Descargando datos de Bitcoin...")
df = yf.download('BTC-USD', start='2016-01-01', end='2026-10-20')

Descargando datos de Bitcoin...


[*********************100%***********************]  1 of 1 completed


In [3]:
returns = 100 * np.log(df['Close'] / df['Close'].shift(1)).dropna()
returns.name = 'BTC_Returns'

In [4]:
# Usamos los datos los primeros 5 años como train inicial.
split_point = 1095 # 365 * 5

# Proxy de volatilidad real (retornos al cuadrado)
real_vol_proxy = returns.iloc[split_point:]**2

In [5]:
semester_len = 182
window_size = split_point  # Tamaño de la ventana de entrenamiento inicial (In-sample)
total_obs = len(returns)

In [6]:
pred_vars_chunks = []
bics_chunks = []
results_list = []

# Avanzamos por bloques del tamaño de un semestre
for i in range(0, total_obs - window_size, semester_len):

    # Tomamos los datos históricos + el semestre que queremos predecir
    # arch necesita ver los datos del test para calcular la volatilidad diaria con los parámetros fijos
    # definimos el indice hasta donde tomar
    current_train_end = window_size + i # indice train, pasa por last_obs en fit
    current_test_end = min(current_train_end + semester_len, total_obs) # indice fin test

    if current_train_end >= total_obs:
        break

    chunk_data = returns.iloc[:current_test_end]

    
    am = arch_model(chunk_data, vol="EGARCH", p=1, q=3, dist="ged")

    try:
        # MAGIA DE ARCH: Entrena SOLAMENTE con los datos previos al semestre actual
        # last_obs utiliza la fecha del último día de entrenamiento disponible
        last_train_date = chunk_data.index[current_train_end]
        res = am.fit(last_obs=last_train_date, disp='off', show_warning=False)
        bics_chunks.append(res.bic)

        # Genera pronósticos de 1 paso adelante para TODO el set de datos provisto
        forecasts = res.forecast(horizon=1, align='target')

        # Extraemos ÚNICAMENTE el bloque que corresponde a este semestre Out-of-Sample
        semester_forecast = forecasts.variance['h.1']
        pred_vars_chunks.append(semester_forecast)

    except Exception as e:
        # Si un semestre falla por convergencia (raro al haber tanta data), continuamos
        continue


# --- Procesamiento de resultados Out-of-Sample ---
try:
    # Unimos todos los semestres predichos en una sola serie continua
    v_pred = pd.concat(pred_vars_chunks)
    v_pred = v_pred.dropna()

    # --- SOLUCIÓN AL ERROR: Intersección de índices ---
    # Buscamos las fechas que están PRESENTES EN AMBOS datasets
    fechas_comunes = v_pred.index.intersection(real_vol_proxy.index)

    # Slicing seguro usando solo las fechas alineadas
    v_pred = v_pred.loc[fechas_comunes]
    v_real = real_vol_proxy.loc[fechas_comunes].squeeze()

    # Tu métrica QLIKE (ahora protegida contra desalineación)
    qlike_series = (v_real / v_pred) - np.log(v_real / v_pred) - 1
    mean_qlike = qlike_series.mean()
    mean_bic = np.mean(bics_chunks)

    results_list.append({
        'Model': "EGARCH",
        'BIC': mean_bic,
        'QLIKE': mean_qlike,
        'QLIKE_Series': qlike_series,
        'Fitted_Res': res
    })

except Exception as e:
    print(f"Error crítico procesando métricas para {model_name}: {e}")



In [7]:
print(results_list[0]["QLIKE"])

1.9942064074085606
